# Gradient Boosted Tree Regression — Databricks Wine Quality

Standalone notebook, one algorithm. Run cells in order. No pandas or RDDs. All metric values shown in explanations are illustrative unless described as mathematical examples. No training has been executed in the author's environment; your notebook prints actual metrics.

## 1. Business problem
Estimate the wine quality score from laboratory chemical measurements.

## 2. Algorithm introduction
Builds trees sequentially to correct the current model errors. For intuition, prediction 5 plus a 0.1 learning-rate correction of 2 becomes 5.2. It is useful for nonlinear tabular prediction.

## 3. Real-world use case and comparison
Random forests average randomized trees; boosting adds sequential corrections. More rounds and depth can overfit.

## 4. Dataset selection and environment
Use `/databricks-datasets/wine-quality/winequality-red.csv`, documented by Databricks and based on the UCI Wine Quality dataset (Cortez et al., 2009). It contains eleven physicochemical features and a sensory quality score. Wine chemistry is not a customer-churn dataset; these notebooks teach reusable modelling patterns. The label is never part of the model input.

The standard red-wine file commonly contains 1,599 rows before deduplication; check the count printed here rather than assuming the workspace copy is identical. Semicolon is the separator. We use the same dataset and deterministic feature-profile split across algorithms, permitting meaningful within-task comparisons.

**Free Edition:** select a recent environment supporting these Spark ML operations. No `sparkContext`, RDD, `cache`, `persist`, or mounts are used. Do not pip-install PySpark. Add MLflow through notebook Environment dependencies if needed. Dataset access and model persistence depend on workspace configuration. If the legacy dataset path is blocked, inspect `/Volumes/samples/databricks/datasets/` for an available copy, or upload the same UCI red-wine CSV to an accessible Unity Catalog volume and set `DATA_PATH_OVERRIDE`. Do not substitute a synthetic file silently. An unavailable Spark ML API requires a supported environment; this notebook does not claim a fallback implementation for every model. Persistence requires a writable catalog/schema and CREATE VOLUME/TABLE plus file permissions.

Sources:
- https://docs.databricks.com/aws/en/getting-started/ml-get-started
- https://docs.databricks.com/aws/en/discover/databricks-datasets
- https://docs.databricks.com/aws/en/compute/serverless/limitations
- https://spark.apache.org/docs/latest/ml-classification-regression.html
- https://spark.apache.org/docs/latest/ml-clustering.html
- https://spark.apache.org/docs/latest/ml-features.html

## 5. Data loading using PySpark

**What and why:** Read the requested real sample data with a reproducible schema contract.

**Functions, important lines and parameters:** `spark.read` builds a CSV reader; header=True uses column names; sep=";" is essential. limit(1).collect() forces an early access check with only one record. toDF normalizes column names. try_cast produces NULL for malformed numeric strings even with ANSI mode. required declares the expected columns; assert catches the wrong dataset. No full dataset is collected. Imports provide Spark transformations, pipeline components, metrics and MLflow; uuid4 later gives unique output paths.

**Expected output and interpretation:** Source path, row count and twelve double columns. Malformed numeric fields become missing values for later handling.

**Common errors and debugging:** PATH_NOT_FOUND or permission denial: verify the source in your workspace and set the override to the same CSV in a permitted volume. One giant column means the separator is wrong. Missing MLflow: add the dependency and restart/run from the top.

In [ ]:
from functools import reduce
from uuid import uuid4
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, ClusteringEvaluator
import mlflow
import math

SEED = 42
DATA_PATH = "/databricks-datasets/wine-quality/winequality-red.csv"
# If blocked, set this to the SAME CSV copied into your accessible UC volume.
DATA_PATH_OVERRIDE = ""
source_path = DATA_PATH_OVERRIDE or DATA_PATH
raw = spark.read.option("header", True).option("sep", ";").csv(source_path)
raw.limit(1).collect()  # Force an early access check; only one row reaches Python.
raw = raw.toDF(*[name.strip().replace(" ", "_") for name in raw.columns])
feature_columns = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH",
    "sulphates", "alcohol"]
required = feature_columns + ["quality"]
assert set(required).issubset(raw.columns), "Check the CSV path and semicolon separator"
numeric = raw.select(*[F.expr(f"try_cast(`{c}` AS DOUBLE)").alias(c) for c in required])
print("Source:", source_path)
print("Raw rows:", numeric.count())
numeric.printSchema()


## 6. Exploratory data analysis
Training-only EDA runs after the split in section 9 so validation/test distributions do not influence feature choices. First we validate types and deterministic domain rules. We will inspect count, mean, standard deviation, minimum/maximum and training label balance. A histogram in Databricks display can reveal unusual ranges. Do not choose transformations by examining test outcomes.

## 7. Data cleaning

**What and why:** Handle invalid values, remove invalid labels and identical full records without learning from held-out data.

**Functions, important lines and parameters:** finite_or_null converts NULL, NaN and infinities to NULL using when/otherwise; feature NULLs will be imputed from training only. between(0,10) and floor restrict quality to integer scores in its domain; missing/invalid labels go to quarantine. dropDuplicates(required) removes only exact duplicate feature+label rows, not all wines with the same score. sha2(to_json(struct(...)),256) identifies each complete record. xxhash64(feature_columns) and pmod(...,100) assign a stable bucket using features only. Identical feature profiles remain in one split, avoiding duplicate leakage.

**Expected output and interpretation:** Rejected-label, duplicate and remaining counts. record_id is a lineage identifier, never a predictor. Hash splitting gives approximate proportions, not exact 70/15/15 counts.

**Common errors and debugging:** Do not fill missing labels with zero. Entirely missing training columns cannot be median-imputed; section 9 checks them. These generic rules do not replace chemistry-specific range review.

In [ ]:
def finite_or_null(name):
    value = F.col(name)
    return F.when(value.isNull() | F.isnan(value) | (F.abs(value) == float("inf")),
                  F.lit(None).cast("double")).otherwise(value)

normalized = numeric.select(*[finite_or_null(c).alias(c) for c in required])
label_valid = F.col("quality").between(0.0, 10.0) & (F.col("quality") == F.floor("quality"))
quarantine = normalized.filter(F.col("quality").isNull() | ~label_valid)
valid = normalized.filter(label_valid)
clean = valid.dropDuplicates(required)
print("Rejected labels:", quarantine.count())
print("Duplicate rows removed:", valid.count() - clean.count())
print("Clean rows:", clean.count())
clean = clean.withColumn("record_id", F.sha2(F.to_json(F.struct(*[F.col(c) for c in required])), 256))
# Identical feature profiles always share a split, even if their labels differ.
clean = clean.withColumn("split_bucket", F.pmod(F.xxhash64(*feature_columns), F.lit(100)))


## 8. Feature engineering

**What and why:** Declare input columns and task output while excluding the answer and identifiers from features.

**Functions, important lines and parameters:** feature_columns contains only eleven chemical inputs. withColumn creates the numerical target label. Imputation, assembly and scaling are defined in section 10 and fitted within each training pipeline. Feature ordering is saved with the model manifest.

**Expected output and interpretation:** A DataFrame ready to split; no model has been fitted.

**Common errors and debugging:** Adding quality, label, record_id or split_bucket to the feature vector creates leakage or meaningless predictors.

In [ ]:
data = clean.withColumn("label", F.col("quality"))


## 9. Train/validation/test split and training EDA

**What and why:** Separate learning, settings selection and final assessment. Inspect training data only.

**Functions, important lines and parameters:** Buckets 0–69 train, 70–84 validate, and 85–99 test. filter defines each subset; count triggers execution and assert catches empty splits. count(column) counts non-NULL training values; first retrieves one summary row. describe reports training distributions. Binary models check both classes in each subset so AUC is defined; only training balance is displayed. Hash splitting suits this independent teaching dataset, not temporal forecasting or known production batch groups.

**Expected output and interpretation:** Three positive split counts and training summary statistics. Counts vary with dataset contents. Scaling statistics never use validation or test.

**Common errors and debugging:** If data is tiny or a class is missing, stop and design an appropriate split. Do not repeatedly resplit until test scores improve.

In [ ]:
train = data.filter(F.col("split_bucket") < 70)
validation = data.filter(F.col("split_bucket").between(70, 84))
test = data.filter(F.col("split_bucket") >= 85)
for name, frame in [("train", train), ("validation", validation), ("test", test)]:
    assert frame.count() > 0, f"Empty {name} split"
    print(name, frame.count())
missing = train.select(*[F.count(c).alias(c) for c in feature_columns]).first().asDict()
assert all(count > 0 for count in missing.values()), "A training feature is entirely null"
train.select(*feature_columns).describe().show(truncate=False)


## 10. Model training

**What and why:** Fit an end-to-end preprocessing and modelling pipeline on training rows only.

**Functions, important lines and parameters:** Imputer(strategy="median") learns training medians. VectorAssembler creates an ordered numeric vector and fails on remaining invalid inputs. StandardScaler(withMean=True,withStd=True) learns training mean/std and makes dense standardized vectors; sensible for eleven features, not massive sparse text. Scaling is essential for distance-based models and retained consistently for all models here. Pipeline fits stages in order; fit(train) returns a PipelineModel. make_pipeline(settings) constructs a fresh pipeline per trial; dictionary expansion combines fixed and candidate settings. maxIter=20 limits boosting rounds; stepSize=0.1 shrinks each update; maxDepth controls tree complexity; seed aids reproducibility.

**Expected output and interpretation:** A fitted pipeline message. No accuracy value is implied. Linear coefficients apply to the preprocessed representation, not raw chemical units.

**Common errors and debugging:** Unsupported ML classes or operations: switch to an appropriate supported environment. Null features: examine imputer inputs and training counts. If optimization fails to converge, inspect scales/data before raising maxIter.

In [ ]:
from pyspark.ml.regression import GBTRegressor
ALGORITHM = '07_gradient_boosted_tree_regression'
TASK = 'regression'
CANDIDATE_SETTINGS = [{'maxDepth': 2}, {'maxDepth': 4}]
FIXED_SETTINGS = {'seed': 42, 'maxIter': 20, 'stepSize': 0.1, 'featuresCol': 'features', 'labelCol': 'label', 'predictionCol': 'prediction'}

def make_pipeline(settings):
    imputed_columns = [c + "_imputed" for c in feature_columns]
    imputer = Imputer(inputCols=feature_columns, outputCols=imputed_columns, strategy="median")
    assembler = VectorAssembler(inputCols=imputed_columns, outputCol="raw_features", handleInvalid="error")
    scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
    estimator = GBTRegressor(**{**FIXED_SETTINGS, **settings})
    return Pipeline(stages=[imputer, assembler, scaler, estimator])

initial_model = make_pipeline(CANDIDATE_SETTINGS[0]).fit(train)
print("Fitted:", ALGORITHM)


## 11. Prediction

**What and why:** Apply the initial model to validation rows without learning anything new.

**Functions, important lines and parameters:** transform executes the fitted imputer, assembler, scaler and model in order. select shows only identifiers and task outputs. limit/show bounds printed output. The PCA output is a vector of coordinates, not a class prediction.

**Expected output and interpretation:** Five validation rows with label and prediction; mismatches are normal.

**Common errors and debugging:** Use the full saved pipeline on raw input columns; do not independently refit a scaler on validation or new data.

In [ ]:
validation_predictions = initial_model.transform(validation)
validation_predictions.select("record_id", "label", "prediction").show(5, truncate=False)


## 12. Model evaluation

**What and why:** Error = actual − prediction. Actual [10,20,30], predicted [12,18,33] gives absolute errors [2,2,3]. MAE=7/3=2.333; MSE=17/3=5.667; RMSE=sqrt(17/3)=2.380; R²=1−17/200=0.915. Lower errors are better. R² can be negative and is not percentage accuracy. Quality is ordinal but treated as numerical for this exercise. MAE is measured in quality-score points; MSE uses squared points.

**Functions, important lines and parameters:** evaluate returns a dictionary. RegressionEvaluator chooses each metric by metricName. Binary evaluation uses when/sum to aggregate TP/TN/FP/FN; first collects only this summary. BinaryClassificationEvaluator uses rawPrediction scores for threshold-independent ranking, not hard classes. ClusteringEvaluator uses standardized features and assignments. PCA uses vector_to_array and higher-order aggregate to sum squared coordinates per row; avg reduces reconstruction error, greatest clips tiny negative roundoff. explainedVariance comes from training PCA, and is labelled accordingly. selection_metric and maximize define model selection direction.

**Expected output and interpretation:** Actual validation metrics printed by your run. No fabricated expected model scores are supplied. Read units and class definitions before comparing results.

**Common errors and debugging:** AUC requires both classes. Silhouette requires nontrivial groups. A constant regression target makes R² uninformative. Do not compare regression RMSE with classification F1.

In [ ]:
def evaluate(frame, fitted=None):
    return {metric: RegressionEvaluator(labelCol="label", predictionCol="prediction",
                    metricName=metric).evaluate(frame) for metric in ["mae", "mse", "rmse", "r2"]}
selection_metric = "rmse"
maximize = False

print("Initial validation metrics:", evaluate(validation_predictions, initial_model))


## 13. Hyperparameter tuning and final test

**What and why:** Compare a small, bounded grid using validation only, then evaluate the chosen model once on the test set.

**Functions, important lines and parameters:** trials stores settings, validation metrics and pipelines. The first fitted model is reused, avoiding duplicate work. Each subsequent trial fits fresh preprocessing on train. min/max chooses the validation criterion; isfinite rejects invalid metrics. The model is not refitted on combined data in this teaching notebook.  Regression compares a training-mean baseline; classification compares training-majority accuracy and prints the confusion counts. K-Means prints per-cluster original-unit feature means. PCA prints training component variance ratios.

**Expected output and interpretation:** Two candidate results, selected settings, and held-out metrics. A candidate can beat the baseline or fail; report the observed result honestly. Cluster feature means describe groups; do not rename them high/low quality.

**Common errors and debugging:** Never tune parameters or thresholds using test results. Comparing algorithms also requires validation-based selection; after choosing among notebooks, use a fresh holdout for an unbiased final project estimate.

In [ ]:
trials = []
for index, settings in enumerate(CANDIDATE_SETTINGS):
    fitted = initial_model if index == 0 else make_pipeline(settings).fit(train)
    scores = evaluate(fitted.transform(validation), fitted)
    assert all(math.isfinite(float(value)) for value in scores.values()), "Invalid metric"
    trials.append({"settings": settings, "metrics": scores, "model": fitted})
    print("Candidate:", settings, scores)
best_trial = (max if maximize else min)(trials, key=lambda trial: trial["metrics"][selection_metric])
best_model = best_trial["model"]
print("Selected:", best_trial["settings"])
# Keep the selected train-fitted pipeline; no refit on test or validation.
test_predictions = best_model.transform(test)
test_metrics = evaluate(test_predictions, best_model)
print("Final test metrics:", test_metrics)
train_mean = train.agg(F.avg("label").alias("mean")).first()["mean"]
baseline_metrics = evaluate(test.withColumn("prediction", F.lit(train_mean)))
print("Training-mean baseline test metrics:", baseline_metrics)


## 14. MLflow experiment tracking

**What and why:** Record comparable trials, their configuration, feature contract and final result.

**Functions, important lines and parameters:** set_tracking_uri selects workspace tracking. current_user supplies your home experiment path. set_experiment creates/selects it. start_run creates one run per trial and a selected-model run; the with block closes each run. log_params records settings, log_metrics numeric results and log_dict a JSON artifact. The selected run includes validation results so model-family comparison need not use test scores. run.info.run_id is lineage used in outputs. Reopening that run later adds model paths. These are tracking records, not registry entries.

**Expected output and interpretation:** Three runs per notebook. Compare algorithms within the same task by validation metrics in the shared task experiment. Exact run IDs are environment-specific.

**Common errors and debugging:** Permission denied: choose a writable experiment directory. Already-active run: finish your unrelated run explicitly. No package: add MLflow through the environment; do not overwrite built-in PySpark.

In [ ]:
mlflow.set_tracking_uri("databricks")
user_name = spark.sql("SELECT current_user() AS u").first()["u"]
experiment_path = f"/Users/{user_name}/databricks_datasets_ml_{TASK}"
mlflow.set_experiment(experiment_path)
for trial in trials:
    with mlflow.start_run(run_name=ALGORITHM + "_candidate"):
        mlflow.log_params({**FIXED_SETTINGS, **trial["settings"], "algorithm": ALGORITHM,
            "source_path": source_path, "split": "feature_hash_70_15_15", "spark_version": spark.version})
        mlflow.log_metrics({"validation_" + k: float(v) for k, v in trial["metrics"].items()})
with mlflow.start_run(run_name=ALGORITHM + "_selected") as run:
    mlflow.log_params({**FIXED_SETTINGS, **best_trial["settings"], "algorithm": ALGORITHM,
        "source_path": source_path, "split": "feature_hash_70_15_15", "spark_version": spark.version})
    mlflow.log_metrics({"validation_" + k: float(v) for k, v in best_trial["metrics"].items()})
    mlflow.log_metrics({"test_" + k: float(v) for k, v in test_metrics.items()})
    mlflow.log_metrics({"baseline_test_" + k: float(v) for k, v in baseline_metrics.items()})
    mlflow.log_dict({"features_in_order": feature_columns, "label_rule":
        "quality >=7" if TASK == "classification" else "quality" if TASK == "regression" else "none"},
        "feature_contract.json")
    selected_run_id = run.info.run_id
print("Experiment:", experiment_path)
print("Run:", selected_run_id)


## 15. Model saving and loading

**What and why:** Persist the entire preprocessing/model pipeline and a Delta source snapshot for later use.

**Functions, important lines and parameters:** current_catalog/current_schema locate your namespace; change CATALOG/SCHEMA before writes if it is read-only. quote_identifier escapes SQL identifiers. CREATE VOLUME IF NOT EXISTS provisions a managed model volume if permitted. uuid4 gives each run an independent directory and table names. data.write.format("delta").mode("errorifexists") avoids overwrites. best_model.write().save stores native Spark PipelineModel stages; PipelineModel.load reloads them. The manifest logs the exact path and feature order to MLflow. In a new session, use the printed absolute model path. Only trusted pipeline artifacts should be loaded.

**Expected output and interpretation:** A managed Delta input snapshot, a model directory and a reloaded PipelineModel. This is native Spark model persistence, not mlflow.spark.log_model and not Unity Catalog model registration.

**Common errors and debugging:** Use a writable non-samples namespace. Insufficient volume permissions require an existing writable volume or admin help. If PipelineModel persistence is unsupported on the selected serverless environment, use a supported environment: saving only coefficients cannot replace these pipelines.

In [ ]:
namespace = spark.sql("SELECT current_catalog() AS c, current_schema() AS s").first()
CATALOG, SCHEMA = namespace.c, namespace.s  # Choose a writable namespace if needed.
VOLUME = "ml_learning_models"
def quote_identifier(value):
    return "`" + value.replace("`", "``") + "`"
prefix = quote_identifier(CATALOG) + "." + quote_identifier(SCHEMA)
volume_identifier = prefix + "." + quote_identifier(VOLUME)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_identifier}")
run_suffix = uuid4().hex[:12]
model_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{ALGORITHM}/{run_suffix}/pipeline"
source_table = prefix + "." + quote_identifier(ALGORITHM + "_source_" + run_suffix)
prediction_table = prefix + "." + quote_identifier(ALGORITHM + "_predictions_" + run_suffix)
data.write.format("delta").mode("errorifexists").saveAsTable(source_table)
best_model.write().save(model_path)
loaded_model = PipelineModel.load(model_path)
with mlflow.start_run(run_id=selected_run_id):
    mlflow.log_params({"model_path": model_path, "source_table": source_table})
    mlflow.log_dict({"model_format": "Spark PipelineModel", "model_path": model_path,
        "source_table": source_table, "features_in_order": feature_columns,
        "source_path": source_path, "algorithm": ALGORITHM}, "model_manifest.json")
print("Model path:", model_path)
print("Source table:", source_table)


## 16. Batch prediction

**What and why:** Apply the reloaded pipeline to a label-free batch and write queryable outputs.

**Functions, important lines and parameters:** select removes quality/label and split metadata. finite_or_null applies the same deterministic cleanup to incoming features. transform applies saved training medians, scaler and model. vector_to_array converts ML vectors to standard arrays for Delta output; classification extracts positive-class probability at index 1, PCA stores component arrays. No vector UDT is written to Delta. withColumn records run ID and timestamp. saveAsTable persists scalar/array columns for SQL consumers.

**Expected output and interpretation:** Five persisted output rows and a unique Delta table name. This demo reuses test features to show inference mechanics; it does not claim they are genuinely new observations.

**Common errors and debugging:** Missing feature names or different units break the contract. All-missing records may technically score after imputation but should be quarantined under a business rule. For production retries, use stable source IDs and model versions with idempotent MERGE logic.

In [ ]:
batch_input = test.select("record_id", *feature_columns)
# Demonstration: held-out rows stand in for a new incoming batch; labels are excluded.
# For real inference replace batch_input with a table having this feature contract.
normalized_batch = batch_input.select("record_id", *[finite_or_null(c).alias(c) for c in feature_columns])
scored = loaded_model.transform(normalized_batch)
output = scored.select("record_id", "prediction")
output = output.withColumn("mlflow_run_id", F.lit(selected_run_id)).withColumn("scored_at", F.current_timestamp())
output.write.format("delta").mode("errorifexists").saveAsTable(prediction_table)
spark.table(prediction_table).show(5, truncate=False)
print("Prediction table:", prediction_table)


## 17. Production considerations
- Preserve the full preprocessing pipeline, feature order, source path/snapshot, code commit, environment and training cutoff. This exercise snapshots data after training for later audit; production should train from an immutable, versioned snapshot created before training.
- Treat sensory quality scores as noisy, subjective labels. The >=7 binary cutoff is illustrative. New regions, vintages and laboratories can shift distributions; this dataset alone does not validate a production quality-control system.
- Split by time/producer/batch when that metadata exists. Identical-feature grouping helps duplicate leakage but cannot discover unknown producer groups.
- Compare with earlier models on identical validation splits. Once you choose a model family, reserve a fresh final holdout; repeatedly consulting the same test set biases evaluation.
- Monitor missing values and feature distributions. For supervised models, join delayed true outcomes and monitor task metrics; for clusters, monitor sizes/stability; for PCA, monitor reconstruction error.
- Retrain on a schedule or validated degradation, evaluate a candidate, retain rollback models, and promote only after quality gates.
- Native Spark pipeline persistence is demonstrated. Registering models additionally needs a supported MLflow model flavor, input/output signature and UC permissions. The JSON manifest is not a serving package. Batch jobs do not need a serving endpoint.
- Separate ingestion, training, validation and scoring job tasks. Move reusable functions into modules after understanding the standalone notebook; test preprocessing contracts, split isolation, metrics and reload parity with pytest. CI should lint/compile and run a small integration job before promotion.
- Free Edition is appropriate for learning, with quota and capability limits; production deployment needs a suitable workspace and operational controls. Small grids and tree counts here conserve resources.

### Three practical exercises
1. Explain `maxDepth` and change only its validation candidate values. Compare validation results before reading test results.
2. Introduce one missing feature value in a training copy. Explain what the training-fitted median imputer does to that value and to a missing inference value.
3. Inspect the saved output table with SQL and trace a prediction/component/cluster back to its MLflow run and model path.

### Five interview questions with answers
1. **What does this algorithm do?** Builds trees sequentially to correct the current model errors. For intuition, prediction 5 plus a 0.1 learning-rate correction of 2 becomes 5.2. It is useful for nonlinear tabular prediction.
2. **Which setting controls the learning behaviour?** maxIter=20 limits boosting rounds; stepSize=0.1 shrinks each update; maxDepth controls tree complexity; seed aids reproducibility.
3. **Why fit preprocessing on training only?** Statistics learned from validation/test would leak information into model selection or final assessment.
4. **How do you evaluate this task?** Error = actual − prediction. Actual [10,20,30], predicted [12,18,33] gives absolute errors [2,2,3]. MAE=7/3=2.333; MSE=17/3=5.667; RMSE=sqrt(17/3)=2.380; R²=1−17/200=0.915. Lower errors are better. R² can be negative and is not percentage accuracy. Quality is ordinal but treated as numerical for this exercise. MAE is measured in quality-score points; MSE uses squared points.
5. **How does it compare with previous methods?** Random forests average randomized trees; boosting adds sequential corrections. More rounds and depth can overfit.

### Troubleshooting scenario
**Symptom:** validation performance looked strong, but loading the saved model gives very different outputs. **Likely cause:** only the estimator was saved while imputation, feature ordering or scaling was rebuilt on new records. **Diagnosis:** compare feature names/order, units, model version and preprocessing parameters on identical input rows. **Fix:** save/load the entire PipelineModel as in section 15 and compare outputs within numerical tolerance.

### Mini assignment
Run this notebook in your workspace. Submit the source path, raw/clean/split counts, two validation trial results, chosen settings, final metrics, experiment run ID and model path. Explain one limitation and compare with an earlier notebook **within the same task**, without choosing based on test results.

### Short quiz
1. Which data should choose hyperparameters: training, validation or test?
2. Can quality be included in the eleven-feature input vector for this task?
3. Does a saved final estimator alone preserve a separately fitted scaler?

Answers: 1. Validation. 2. No; it is the supervised outcome or intentionally excluded from the unsupervised task. 3. No; save the full pipeline.

Pause here. You can create all notebooks now while studying them one at a time; do not move to the next lesson until you are ready.